In [ ]:
import re
import os
import json
import calendar
from dateutil import parser
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

import requests
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from scipy.interpolate import RBFInterpolator
from scipy.interpolate import PchipInterpolator

from api_client import TradingDeskAPI
from options import OptionSurface, Deribit, OKX, Bybit

In [ ]:
load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
print(BASE_URL)

target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

s = OptionSurface()

deribit = Deribit(currencies=["BTC", "ETH"], target_expiry=target_expiry)
okx = OKX(currencies=["BTC", "ETH"], target_expiry=target_expiry)
bybit = Bybit(currencies=["BTC", "ETH"], target_expiry=target_expiry)

s.initialize(currencies=["BTC", "ETH"], exchanges=[deribit, okx, bybit])

https://alphasignal-dev.moretoncp.com
spot: 64841.0 volume24h: 0.2407
spot: 1918.0 volume24h: 4.6843
spot: 64859.1 volume24h: 976.83772218
spot: 1918.99 volume24h: 16305.612738
spot: 64851.4 volume24h: 1601.312729
spot: 1918.84 volume24h: 16285.24319


In [ ]:
api = TradingDeskAPI(BASE_URL, USER_EMAIL, USER_PASSWORD)

markets = api.get_markets(limit=10000, liquidity_num_min=10000, volume_num_min=5000)

markets_df = pd.DataFrame(markets)

markets_df.head()

,id,question,conditionId,slug,endDate,liquidity,startDate,image,icon,description,...,gameStartTime,secondsDelay,positionIds,sportsMarketType,marketMetadata,gameId,eventStartTime,oneYearPriceChange,umaResolutionStatus,line
0,1321561,Will SpaceX have the highest IPO Market Cap 2026?,0x858b982b9ea15f0133c1ae8c53a5b7bbdf67891eca22...,will-spacex-have-the-highest-ipo-market-cap-20...,2026-12-31T00:00:00Z,99976.7364,2026-02-02T22:34:41.427627Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,This market will resolve to the company that a...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,561262,Will James Talarico win the 2028 US Presidenti...,0x9eb7ac1524fdacb46ceb8758454f0004c7b117eae76e...,will-james-talarico-win-the-2028-us-presidenti...,2028-11-07T00:00:00Z,998560.85025,2025-07-11T19:06:20.546Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,The 2028 US Presidential Election is scheduled...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,665374,Will the U.S. invade Iran before 2027?,0x5db999fad322cea2914535aae5517060c3f80ad6d8c0...,will-the-us-invade-iran-before-2027,2026-12-31T00:00:00Z,995919.768,2025-11-05T17:52:17.414Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will resolve to ""Yes"" if the Unite...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3257334,"Will Bitcoin reach $85,000 in August?",0x7f668805bfee7909997562f60489ddef01e40a3dca43...,will-bitcoin-reach-85k-in-august-2026,2026-09-01T04:00:00Z,99427.76479,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,2027-01-01T05:00:00Z,99423.27245,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
BTC_KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt",
    "bitcoin price",
    "bitcoin hits",
    "bitcoin above",
    "bitcoin below",
]

ETH_KEYWORDS = [
    "ethereum",
    "eth",
    "ethereum price",
    "ethereum hits",
    "ethereum above",
    "ethereum below",
]

def is_btc_market(row):
    text = (str(row["question"]) + " " + str(row.get("description", ""))).lower()

    return any(k in text for k in BTC_KEYWORDS)

btc_df = markets_df[markets_df.apply(is_btc_market, axis=1)]
btc_df

,id,question,conditionId,slug,endDate,liquidity,startDate,image,icon,description,...,gameStartTime,secondsDelay,positionIds,sportsMarketType,marketMetadata,gameId,eventStartTime,oneYearPriceChange,umaResolutionStatus,line
3,3257334,"Will Bitcoin reach $85,000 in August?",0x7f668805bfee7909997562f60489ddef01e40a3dca43...,will-bitcoin-reach-85k-in-august-2026,2026-09-01T04:00:00Z,99427.76479,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,2027-01-01T05:00:00Z,99423.27245,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,2027-01-01T05:00:00Z,96952.3725,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will resolve to ""Yes"" if any Binan...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,701502,"Will Bitcoin dip to $45,000 by December 31, 2026?",0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,2027-01-01T05:00:00Z,95301.0835,2025-11-24T19:07:15.386Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
51,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,2027-01-01T05:00:00Z,91871.6955,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59,3257368,"Will Bitcoin dip to $42,500 in August?",0x98e5997750cfc27c47ad3b4f4278a01935bed1edfcec...,will-bitcoin-dip-to-42pt5k-in-august-2026,2026-09-01T04:00:00Z,90742.32489,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,2026-09-01T04:00:00Z,88031.64137,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
btc_df = btc_df[
            (btc_df["acceptingOrders"] == True) &
            (btc_df["enableOrderBook"] == True)
        ]

btc_df["tokens"] = btc_df["clobTokenIds"].apply(json.loads)
btc_df["yes_token"] = btc_df["tokens"].apply(lambda x: x[0])
btc_df["no_token"] = btc_df["tokens"].apply(lambda x: x[1])

btc_df

,id,question,conditionId,slug,endDate,liquidity,startDate,image,icon,description,...,sportsMarketType,marketMetadata,gameId,eventStartTime,oneYearPriceChange,umaResolutionStatus,line,tokens,yes_token,no_token
3,3257334,"Will Bitcoin reach $85,000 in August?",0x7f668805bfee7909997562f60489ddef01e40a3dca43...,will-bitcoin-reach-85k-in-august-2026,2026-09-01T04:00:00Z,99427.76479,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[112241751071174396487292868128075924275858867...,1122417510711743964872928681280759242758588672...,7643548942136022074918172869662533578449021356...
4,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,2027-01-01T05:00:00Z,99423.27245,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[788132865028612870351706755236880554943981441...,7881328650286128703517067552368805549439814411...,1756428523402968434262418461776029763290473809...
21,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,2027-01-01T05:00:00Z,96952.3725,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will resolve to ""Yes"" if any Binan...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[955325244079599031455105836849790497350285024...,9553252440795990314551058368497904973502850245...,1078699501271841417437724297387015787237705826...
33,701502,"Will Bitcoin dip to $45,000 by December 31, 2026?",0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,2027-01-01T05:00:00Z,95301.0835,2025-11-24T19:07:15.386Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[989451065105237308840806703103151321280161853...,9894510651052373088408067031031513212801618531...,2032639479870041357180166074943703627262720347...
51,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,2027-01-01T05:00:00Z,91871.6955,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[596839742467995258312442199211144557362757050...,5968397424679952583124421992111445573627570503...,7012278860989242444626087945307690553239457130...
59,3257368,"Will Bitcoin dip to $42,500 in August?",0x98e5997750cfc27c47ad3b4f4278a01935bed1edfcec...,will-bitcoin-dip-to-42pt5k-in-august-2026,2026-09-01T04:00:00Z,90742.32489,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[108302454607081176309311291655387968923267938...,1083024546070811763093112916553879689232679381...,9456528654693780118273015372788905703522251986...
86,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,2026-09-01T04:00:00Z,88031.64137,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[747390749095846764977874016868928588526100638...,7473907490958467649778740168689285885261006384...,927704830981350751459269

In [7]:
def get_books(row):

    yes_book = api.get_orderbook(row.yes_token)
    no_book = api.get_orderbook(row.no_token)

    return pd.Series({
        "yes_book": yes_book,
        "no_book": no_book,

        "yes_bid": float(yes_book["bids"][-1]["price"]) if yes_book["bids"] else None,
        "yes_ask": float(yes_book["asks"][-1]["price"]) if yes_book["asks"] else None,

        "no_bid": float(no_book["bids"][-1]["price"]) if no_book["bids"] else None,
        "no_ask": float(no_book["asks"][-1]["price"]) if no_book["asks"] else None,
    })

with ThreadPoolExecutor(max_workers=30) as executor:
    results = list(executor.map(get_books, [row for _, row in btc_df.iterrows()]))

btc_df[
    [
        "yes_book",
        "no_book",
        "yes_bid",
        "yes_ask",
        "no_bid",
        "no_ask"
    ]
] = pd.DataFrame(results).values

In [8]:
def extract_strike(question):

    match = re.search(
        r'\$([\d,]+)',
        question
    )

    if match:
        return float(
            match.group(1).replace(",", "")
        )

    return None

def extract_expiry(question, slug=None):
    # Try full date in question
    # m = match
    m = re.search(
        r'(January|February|March|April|May|June|July|August|September|October|November|December) \d{1,2}, \d{4}',
        question
    )
    if m:
        return parser.parse(m.group()).strftime("%Y%m%d")

    if slug:
        slug = slug.lower()

        # Full date in slug: december-31-2026
        m = re.search(
            r'(january|february|march|april|may|june|july|august|september|october|november|december)-(\d{1,2})-(\d{4})',
            slug
        )
        if m:
            month, day, year = m.groups()
            return parser.parse(f"{day} {month} {year}").strftime("%Y%m%d")

        # Month only: august-2026
        m = re.search(
            r'(january|february|march|april|may|june|july|august|september|october|november|december)-(\d{4})',
            slug
        )
        if m:
            month_name, year = m.groups()
            month = parser.parse(month_name).month
            year = int(year)

            last_day = calendar.monthrange(year, month)[1]
            return f"{year}{month:02d}{last_day:02d}"

    return None

btc_df["strike"] = btc_df.question.apply(
    extract_strike
)

btc_df["expiry"] = btc_df.apply(
    lambda row: extract_expiry(row["question"], row["slug"]),
    axis=1,
)

In [9]:
def parse_btc_market_type(question):

    q = question.lower()

    # ----------------------
    # Direction
    # ----------------------
    if any(word in q for word in [
        "dip",
        "fall",
        "drop",
        "below",
        "under",
        "crash"
    ]):
        direction = "down"

    elif any(word in q for word in [
        "reach",
        "hit",
        "touch",
        "above",
        "over",
        "exceed"
    ]):
        direction = "up"

    else:
        direction = None

    # ----------------------
    # Event type
    # ----------------------
    if any(word in q for word in [
        "reach",
        "hit",
        "touch",
        "dip to",
        "fall to",
        "drop to",
        "crash to"
    ]):
        event_type = "touch"

    elif any(word in q for word in [
        "be above",
        "above on",
        "close above",
        "be below",
        "below on",
        "close below",
        "finish above",
        "finish below"
    ]):
        event_type = "expiry"


    else:
        event_type = None

    return {
        "direction": direction,
        "event_type": event_type
    }

btc_df[
    ["direction", "event_type"]
] = btc_df["question"].apply(
    lambda x: pd.Series(parse_btc_market_type(x))
)

In [12]:
btc_df["expiry_dt"] = pd.to_datetime(btc_df["expiry"], format="%Y%m%d", utc=True)
btc_df["T"] = (btc_df["expiry_dt"] - datetime.now(timezone.utc)).dt.total_seconds() / (365.25 * 24 * 3600)
btc_df = btc_df[btc_df["T"] > 0]

In [13]:
btc_df

,id,question,conditionId,slug,endDate,liquidity,startDate,image,icon,description,...,yes_bid,yes_ask,no_bid,no_ask,strike,expiry,direction,event_type,expiry_dt,T
3,3257334,"Will Bitcoin reach $85,000 in August?",0x7f668805bfee7909997562f60489ddef01e40a3dca43...,will-bitcoin-reach-85k-in-august-2026,2026-09-01T04:00:00Z,99427.76479,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.006,0.007,0.993,0.994,85000.0,20260831,up,touch,2026-08-31 00:00:00+00:00,0.059441
4,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,2027-01-01T05:00:00Z,99423.27245,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.018,0.02,0.98,0.982,5000.0,20261231,down,touch,2026-12-31 00:00:00+00:00,0.393459
21,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,2027-01-01T05:00:00Z,96952.3725,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will resolve to ""Yes"" if any Binan...",...,0.72,0.73,0.27,0.28,70000.0,20261231,up,touch,2026-12-31 00:00:00+00:00,0.393459
33,701502,"Will Bitcoin dip to $45,000 by December 31, 2026?",0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,2027-01-01T05:00:00Z,95301.0835,2025-11-24T19:07:15.386Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.23,0.24,0.76,0.77,45000.0,20261231,down,touch,2026-12-31 00:00:00+00:00,0.393459
51,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,2027-01-01T05:00:00Z,91871.6955,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.032,0.039,0.961,0.968,20000.0,20261231,down,touch,2026-12-31 00:00:00+00:00,0.393459
59,3257368,"Will Bitcoin dip to $42,500 in August?",0x98e5997750cfc27c47ad3b4f4278a01935bed1edfcec...,will-bitcoin-dip-to-42pt5k-in-august-2026,2026-09-01T04:00:00Z,90742.32489,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.009,0.01,0.99,0.991,42500.0,20260831,down,touch,2026-08-31 00:00:00+00:00,0.059441
86,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,2026-09-01T04:00:00Z,88031.64137,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.022,0.023,0.977,0.978,50000.0,20260831,down,touch,2026-08-31 00:00:00+00:00,0.059441


In [14]:
def calculate_ev(
        model_prob,
        best_ask_yes,
        best_bid_yes,
        best_ask_no,
        best_bid_no,
        fee_rate=0.07,
        kelly_fraction = 0.5
    ):
    
    """
    Expected PnL per contract for Polymarket.

    model_prob : P(YES)
    fee_rate   : taker fee rate (e.g. 0.07 for crypto markets)

    Assumes:
      - you are a taker
      - fee = fee_rate * price * (1 - price)
      - settlement has no fee
    """

    def is_valid(price):
        return pd.notna(price)

    p_yes = model_prob
    p_no = 1 - p_yes

    def fee(price):
        #fee = C × feeRate × p × (1 - p)
        #Where C = number of shares traded and p = price of the shares.
        return fee_rate * price * (1 - price)

    # -------------------------
    # BUY YES
    # -------------------------
    if is_valid(best_ask_yes):
        buy_yes_cost = best_ask_yes + fee(best_ask_yes)

        profit_if_yes = 1 - buy_yes_cost
        cost_if_no = buy_yes_cost

        # EV: profit if yes - cost if no
        buy_yes_ev = p_yes * profit_if_yes - p_no * cost_if_no

        """
        p            : probability of winning
        win_profit   : net profit if the bet wins
        loss_amount  : net loss if the bet loses

        Returns the optimal Kelly fraction.
        """

        buy_yes_kelly = kelly_fraction * (buy_yes_ev / profit_if_yes) # fee adjusted kelly

        # -------------------------
        # SELL YES (short YES)
        # -------------------------
        sell_yes_credit = best_bid_yes - fee(best_bid_yes)

        profit_if_no = sell_yes_credit
        cost_if_yes = 1 - sell_yes_credit # need to pay the remaining of the $1 out of the credit you got, if yes happened

        # EV: profit if yes - cost if no
        sell_yes_ev = p_no * profit_if_no - p_yes * cost_if_yes

        sell_yes_kelly = kelly_fraction * (sell_yes_ev / profit_if_no)

    else:
        buy_yes_ev = np.nan
        sell_yes_ev = np.nan
        buy_yes_kelly = np.nan
        sell_yes_kelly = np.nan
        
    # -------------------------
    # BUY NO
    # -------------------------
    if is_valid(best_ask_no):
        buy_no_cost = best_ask_no + fee(best_ask_no)

        profit_if_no = 1 - buy_no_cost
        cost_if_yes = buy_no_cost

        buy_no_ev = p_no * profit_if_no - p_yes * cost_if_yes

        buy_no_kelly = kelly_fraction * (buy_no_ev / profit_if_no)

        # -------------------------
        # SELL NO (short NO)
        # -------------------------
        sell_no_credit = best_bid_no - fee(best_bid_no)

        profit_if_yes = sell_no_credit
        cost_if_no = 1 - sell_no_credit

        sell_no_ev = p_yes * profit_if_yes - p_no * cost_if_no

        sell_no_kelly = kelly_fraction * (sell_no_ev / profit_if_yes)

    else:
        buy_no_ev = np.nan
        sell_no_ev = np.nan
        buy_no_kelly = np.nan
        sell_no_kelly = np.nan

    print("buy_yes_ev:", buy_yes_ev)
    print("sell_yes_ev:", sell_yes_ev)
    print("buy_no_ev:", buy_no_ev)
    print("sell_no_ev:", sell_no_ev)
    print("buy_yes_kelly:", buy_yes_kelly)
    print("sell_yes_kelly:", sell_yes_kelly)
    print("buy_no_kelly:", buy_no_kelly)
    print("sell_no_kelly:", sell_no_kelly)

    return {
        "buy_yes_fee": fee(best_ask_yes),
        "sell_yes_fee": fee(best_bid_yes),
        "buy_no_fee": fee(best_ask_no),
        "sell_no_fee": fee(best_bid_no),
        "buy_yes_ev": buy_yes_ev,
        "sell_yes_ev": sell_yes_ev,
        "buy_no_ev": buy_no_ev,
        "sell_no_ev": sell_no_ev,
        "buy_yes_kelly": buy_yes_kelly,
        "sell_yes_kelly": sell_yes_kelly,
        "buy_no_kelly": buy_no_kelly,
        "sell_no_kelly": sell_no_kelly,
    }


result = calculate_ev(
    model_prob=0.059,
    best_ask_yes=0.1,
    best_bid_yes=0.09,
    best_ask_no=0.91,
    best_bid_no=0.9,
    fee_rate=0.07
)

buy_yes_ev: -0.04730000000000002
sell_yes_ev: 0.025266999999999998
buy_no_ev: 0.025266999999999984
sell_no_ev: -0.04729999999999996
buy_yes_kelly: -0.026463018910148833
sell_yes_kelly: 0.14992227087709303
buy_no_kelly: 0.14992227087709298
sell_no_kelly: -0.026463018910148794


In [15]:
def calculate_market_ev(row, s, confidence=0.7):

    currency = "BTC"
    required_strike = row["strike"]

    T = row["T"]
    fee_rate = row["feeSchedule"]["rate"]

    iv = s.get_iv_from_surface(exchange=s, currency=currency, required_strike=required_strike, T=T)
    print('iv', iv)

    if iv is None:
        return None

    if row["event_type"] == "touch":

        p_touch_above, p_touch_below = s.prob_touch(
                    spot=s.data[currency]["weighted_spot"],
                    required_strike=required_strike,
                    iv=iv,
                    T=T,
                    r=0)
        
        if row["direction"] == "up":
            model_prob = p_touch_above

        elif row["direction"] == "down":
            model_prob = p_touch_below

    if row["event_type"] == "expiry":

        p_finish_above, p_finish_below = s.prob_finish(
                    spot=s.data[currency]["weighted_spot"], 
                    required_strike=required_strike, 
                    iv=iv,
                    T=T,
                    r=0)

        if row["direction"] == "up":
            model_prob = p_finish_above

        elif row["direction"] == "down":
            model_prob = p_finish_below

    market_prob = (row["yes_bid"] + row["yes_ask"]) / 2
    model_prob_adjusted = (confidence * model_prob + (1-confidence) * market_prob)

    ev = calculate_ev(
        model_prob=model_prob,
        best_ask_yes=row["yes_ask"],
        best_bid_yes=row["yes_bid"],
        best_ask_no=row["no_ask"],
        best_bid_no=row["no_bid"],
        fee_rate=fee_rate
    )

    return pd.Series({
        "iv": iv,
        "model_prob": model_prob,
        **ev
    })

btc_df[["iv", "model_prob", "buy_yes_ev", "sell_yes_ev", "buy_no_ev", "sell_no_ev",
        "buy_yes_fee", "sell_yes_fee", "buy_no_fee", "sell_no_fee",
        "buy_yes_kelly", "sell_yes_kelly", "buy_no_kelly", "sell_no_kelly"]] = btc_df.apply(
    lambda row: calculate_market_ev(row, s),
    axis=1,
)

iv 0.42346501445053397
p_touch_above: 0.00694 p_touch_below: 1.0
buy_yes_ev: -0.0005465700000000006
sell_yes_ev: -0.0013574799999999995
buy_no_ev: -0.0013574799999999648
sell_no_ev: -0.0005465699999999702
buy_yes_kelly: -0.00027534640009858636
sell_yes_kelly: -0.12158308434183841
buy_no_kelly: -0.12158308434183453
sell_no_kelly: -0.00027534640009857107
iv 0.5884254072224595
p_touch_above: 1.0 p_touch_below: 0.0
buy_yes_ev: -0.021372000000000002
sell_yes_ev: 0.01676268
buy_no_ev: 0.01676268000000003
sell_no_ev: -0.021372000000000058
buy_yes_kelly: -0.010919368748901525
sell_yes_kelly: 0.5
buy_no_kelly: 0.5
sell_no_kelly: -0.010919368748901554
iv 0.3918172875667704
p_touch_above: 0.70307 p_touch_below: 1.0
buy_yes_ev: -0.04072699999999996
sell_yes_ev: 0.002817999999999987
buy_no_ev: 0.002817999999999987
sell_no_ev: -0.04072700000000004
buy_yes_kelly: -0.0794818952159029
sell_yes_kelly: 0.001996067364794406
buy_no_kelly: 0.001996067364794406
sell_no_kelly: -0.07948189521590308
iv 0.505363

In [16]:
btc_df

,id,question,conditionId,slug,endDate,liquidity,startDate,image,icon,description,...,buy_no_ev,sell_no_ev,buy_yes_fee,sell_yes_fee,buy_no_fee,sell_no_fee,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly
3,3257334,"Will Bitcoin reach $85,000 in August?",0x7f668805bfee7909997562f60489ddef01e40a3dca43...,will-bitcoin-reach-85k-in-august-2026,2026-09-01T04:00:00Z,99427.76479,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.000417,0.000487,-0.000547,-0.001357,-0.001357,-0.000547,-0.000275,-0.121583,-0.121583,-0.000275
4,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,2027-01-01T05:00:00Z,99423.27245,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.001237,0.001372,-0.021372,0.016763,0.016763,-0.021372,-0.010919,0.500000,0.500000,-0.010919
21,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,2027-01-01T05:00:00Z,96952.3725,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will resolve to ""Yes"" if any Binan...",...,0.014112,0.013797,-0.040727,0.002818,0.002818,-0.040727,-0.079482,0.001996,0.001996,-0.079482
33,701502,"Will Bitcoin dip to $45,000 by December 31, 2026?",0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,2027-01-01T05:00:00Z,95301.0835,2025-11-24T19:07:15.386Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.012397,0.012768,0.031732,-0.066897,-0.066897,0.031732,0.021233,-0.153713,-0.153713,0.021233
51,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,2027-01-01T05:00:00Z,91871.6955,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.002168,0.002624,0.006546,-0.018338,-0.018338,0.006546,0.003415,-0.307363,-0.307363,0.003415
59,3257368,"Will Bitcoin dip to $42,500 in August?",0x98e5997750cfc27c47ad3b4f4278a01935bed1edfcec...,will-bitcoin-dip-to-42pt5k-in-august-2026,2026-09-01T04:00:00Z,90742.32489,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.000624,0.000693,-0.002703,0.000386,0.000386,-0.002703,-0.001366,0.023023,0.023023,-0.001366
86,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,2026-09-01T04:00:00Z,88031.64137,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.001506,0.001573,0.015117,-0.019196,-0.019196,0.015117,0.007749,-0.468338,-0.468338,0.007749


In [17]:
btc_df = btc_df.dropna(subset=["buy_yes_ev"])

ev_cols = [
    "buy_yes_ev",
    "sell_yes_ev",
    "buy_no_ev",
    "sell_no_ev"
]

btc_df["best_ev"] = btc_df[ev_cols].max(axis=1)

btc_df["best_action"] = btc_df[ev_cols].idxmax(axis=1)

In [18]:
opportunities = btc_df[btc_df["best_ev"] > 0].sort_values("best_ev", ascending=False)

opportunities

,id,question,conditionId,slug,endDate,liquidity,startDate,image,icon,description,...,buy_yes_fee,sell_yes_fee,buy_no_fee,sell_no_fee,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
21,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,2027-01-01T05:00:00Z,96952.3725,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will resolve to ""Yes"" if any Binan...",...,-0.040727,0.002818,0.002818,-0.040727,-0.079482,0.001996,0.001996,-0.079482,0.014112,sell_yes_ev
33,701502,"Will Bitcoin dip to $45,000 by December 31, 2026?",0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,2027-01-01T05:00:00Z,95301.0835,2025-11-24T19:07:15.386Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.031732,-0.066897,-0.066897,0.031732,0.021233,-0.153713,-0.153713,0.021233,0.012768,buy_yes_ev
51,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,2027-01-01T05:00:00Z,91871.6955,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.006546,-0.018338,-0.018338,0.006546,0.003415,-0.307363,-0.307363,0.003415,0.002624,sell_no_ev
86,3257362,"Will Bitcoin dip to $50,000 in August?",0xdf1463edbf062d8fdd82c5eb0bdb5fd69450b0c21c9d...,will-bitcoin-dip-to-50k-in-august-2026,2026-09-01T04:00:00Z,88031.64137,2026-08-01T05:07:37Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,0.015117,-0.019196,-0.019196,0.015117,0.007749,-0.468338,-0.468338,0.007749,0.001573,sell_no_ev
4,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,2027-01-01T05:00:00Z,99423.27245,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,-0.021372,0.016763,0.016763,-0.021372,-0.010919,0.500000,0.500000,-0.010919,0.001372,sell_no_ev
59,3257368,"Will Bitcoin dip to $42,500 in August?",0x98e5997750cfc27c47ad3b4f4278a01935bed1edfcec...,will-bitcoin-dip-to-42pt5k-in-august-2026,2026-09-01T04:00:00Z,90742.32489,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,-0.002703,0.000386,0.000386,-0.002703,-0.001366,0.023023,0.023023,-0.001366,0.000693,sell_no_ev
3,3257334,"Will Bitcoin reach $85,000 in August?",0x7f668805bfee7909997562f60489ddef01e40a3dca43...,will-bitcoin-reach-85k-in-august-2026,2026-09-01T04:00:00Z,99427.76479,2026-08-01T05:07:39Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,"This market will immediately resolve to ""Yes"" ...",...,-0.000547,-0.001357,-0.001357,-0.000547,-0.000275,-0.121583,-0.121583,-0.000275,0.000487,sell_no_ev


In [ ]:
# get all dfs

orders_df = pd.read_parquet("orders.parquet")
fills_df = pd.read_parquet("fills.parquet")
positions_df = pd.read_parquet("positions.parquet")
realized_pnl_df = pd.read_parquet("realized_pnl.parquet")
equity_df = pd.read_parquet("equity.parquet")

In [ ]:
# Inventory Management Step

# buys = trades_df[trades_df.action == "BUY"]
# sells = trades_df[trades_df.action == "SELL"]

# position = (buys.shares.sum() - sells.shares.sum())

EXIT_THRESHOLD = -0.02
 
positions = api.get_positions()

for pos in positions:

    condition_id = pos["condition_id"]
    row = btc_df.loc[btc_df["condition_id"] == condition_id]

    if pos.outcome == "Yes": # no shorting in polymarket
        sell_price = row.yes_bid
        hold_ev = row.buy_yes_ev
        exit_ev = row.sell_yes_ev

    elif pos.outcome == "No":
        sell_price = row.no_bid
        hold_ev = row.buy_no_ev
        exit_ev = row.sell_no_ev

    if hold_ev < EXIT_THRESHOLD:
        best_action = "exit"
        api.close_position(pos)

    else:
        best_action = "hold"

    size = positions_df["size"]

    entry_price = pos["entry_price"]

    realized_pnl = (sell_price - entry_price) * size

    positions_df.remove(condition_id)

    realized_pnl_df += realized_pnl

    trades_df.append(entry_price, realized_pnl, size, exit_ev)

    print("question:", row.question,
        "pos.outcome:", pos.outcome,
        "pos.size", pos.size,
        "exit_threshold:", EXIT_THRESHOLD,
        "hold_ev", hold_ev,
        "exit_ev", exit_ev,
        "best_action", best_action,
        "condition_id", condition_id
    )

In [ ]:
def close_position(condition_id):

    positions = api.get_positions()

    pos = positions.find(condition_id)

    order = api.close_position(pos)

    # order_id
    # condition_id
    # token_id
    # outcome
    # side
    # price
    # requested_size
    # status
    # created_at
    # cancelled_at

    # order  = {"name": "Charlie", "age": 35}
    orders_df.loc[len(orders_df)] = order

    print("placed order to close position")

In [ ]:
# fill_id
# order_id
# condition_id
# token_id
# outcome
# side
# price
# shares
# fee
# timestamp
fill = {}

fills_df.loc[len(fills_df)] = fill

position = sum(
    fills_df.shares if fills_df.side == "BUY" else -fills_df.shares
    for fill in fills_df
)

In [ ]:
# fill_id
# condition_id
# token_id
# outcome
# shares
# entry_price
# exit_price
# gross_pnl
# fees
# net_pnl
# entry_time
# exit_time
# holding_time

realized_pnl_df.loc[len(realized_pnl_df)] = row

In [ ]:
# timestamp
# cash
# market_value
# equity
# realized_pnl
# unrealized_pnl
# daily_return

# return_t = equity_t / equity_{t-1} - 1

equity_df.loc[len(equity_df)] = row

In [ ]:
# New Opportunities Step

MIN_EV = 0.01   # require 1% edge, default = 0
bankroll = 100
MAX_POSITION = 0.05

positions = []

for _, trade in opportunities.iterrows():

    if trade.best_ev < MIN_EV:
        continue

    # usually cannot short, this only happens when im closing positions
    if trade.best_action == "buy_yes_ev" or trade.best_action == "sell_no_ev":

        token = trade.yes_token
        side = "YES"
        kelly = trade.buy_yes_kelly
        buy_price = trade.yes_ask

    elif trade.best_action == "buy_no_ev" or trade.best_action == "sell_yes_ev":

        token = trade.no_token
        side = "NO"
        kelly = trade.buy_no_kelly
        buy_price = trade.no_ask

    dollars = bankroll * kelly

    dollars = min(dollars, bankroll * MAX_POSITION)

    position = api.get_conditional_balance(token)

    current_size = float(position["balance"])

    # inventory cap
    dollars = min(dollars, bankroll * MAX_POSITION - current_size * buy_price)

    if dollars <= 0:
        continue

    size = dollars / buy_price

    normalized_size = api.normalize_size(size)

    api.place_limit_order(
        side=side,
        shares=normalized_size,
        price=buy_price
    )

    total_size = current_size + normalized_size

    new_entry_price = (current_size * entry_price + normalized_size * buy_price) / total_size

    fills_df.append({
        "condition_id": trade.conditionId,
        "token_id": token_id,
        "outcome": side,
        "size": shares,
        "entry_price": price,
        "entry_time": datetime.now()
    })

    fills.loc[
    fills.token_id == token,
    "shares"
] += new_shares




In [ ]:
def place_limit(condition_id):

    positions = api.get_positions()

    pos = positions.find(condition_id)

    api.place_limit_order(
        side=side,
        shares=normalized_size,
        price=buy_price
    )

    entry_price = pos["entry_price"]
    
    total_size = current_size + normalized_size
    
    new_entry_price = (current_size * entry_price + normalized_size * buy_price) / total_size

    trades_df.append(new_entry_price, normalized_size, buy_ev)

    positions[condition_id] = new_entry_price, total_size

    print("placed limit order")

In [ ]:
def save(df):

    date = datetime.now().strftime("%Y%m%d")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    DATA_DIR = f"data/{date}"
    os.makedirs(DATA_DIR, exist_ok=True)
    filename = f"{DATA_DIR}/markets_{timestamp}.parquet"

    df.to_parquet(filename, engine="fastparquet", index=False)

    print('df saved at: ', filename)

save(btc_df)

In [ ]:
df = []

df.append({
        "condition_id": 1,
        "token_id": "nan",
        "outcome": "YES",
        "size": 1.0,
        "entry_price": 0.40,
        "entry_time": datetime.now(),
        "mtm_pnl": 0.0,
        "realized_pnl": 0.0
    })

df = pd.DataFrame(df)

DATA_DIR = f"data"
os.makedirs(DATA_DIR, exist_ok=True)
filename = f"{DATA_DIR}/positions.parquet"

df.to_parquet(filename, engine="fastparquet", index=False)

filename = f"{DATA_DIR}/trades.parquet"

df.to_parquet(filename, engine="fastparquet", index=False)

print('df saved at: ', filename)

AttributeError: 'list' object has no attribute 'to_parquet'

In [ ]:
ENTRY_THRESHOLD = 0.03
EXIT_THRESHOLD = 0.01

edge = 0.031

if edge > ENTRY_THRESHOLD:
    pass
    #sell_yes()

if edge < EXIT_THRESHOLD:
    pass
    #close_position()

In [ ]:
#                  Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
while True:

    # 1. Manage inventory
    positions = api.get_positions()

    for pos in positions:

        market = model.get_market(pos.condition_id)

        if pos.outcome == "Yes":
            hold_ev = market.buy_yes_ev
            exit_ev = market.sell_yes_ev

        else:
            hold_ev = market.buy_no_ev
            exit_ev = market.sell_no_ev

        if hold_ev < EXIT_THRESHOLD:
            api.close_position(pos)


    # 2. Find new opportunities
    opportunities = scan_markets()

    for trade in opportunities:

        if trade.best_ev > MIN_EV:
            api.place_limit_order(...)


In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4

In [ ]:
{'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 'slug': 'will-bitcoin-reach-100000-by-december-31-2026-571-361-361', 
 'resolutionSource': '', 'endDate': '2027-01-01T05:00:00Z', 'liquidity': '94083.1351', 'startDate': '2025-11-24T19:07:17.691Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for Bitcoin (BTC/USDT) between November 24, 2025, 14:00 and December 31, 2026, 23:59 in the ET timezone has a final "High" price equal to or greater than the price specified in the title. Otherwise, this market will resolve to "No."\n\nThe resolution source for this market is Binance, specifically the BTC/USDT "High" prices available at https://www.binance.com/en/trade/BTC_USDT, with the chart settings on "1m" for one-minute candles selected on the top bar.\n\nPlease note that the outcome of this market depends solely on the price data from the Binance BTC/USDT trading pair. Prices from other exchanges, different trading pairs, or spot markets will not be considered for the resolution of this market.', 
'outcomes': '["Yes", "No"]', 'outcomePrices': '["0.095", "0.905"]', 'volume': '2370982.5583320004', 'active': True, 'closed': False, 'marketMakerAddress': '', 'createdAt': '2025-11-24T18:55:12.725029Z', 'updatedAt': '2026-08-02T10:54:52.526927Z', 
'new': False, 'featured': False, 'submitted_by': '0x91430CaD2d3975766499717fA0D66A78D814E5c5', 'archived': False, 'resolvedBy': '0x65070BE91477460D8A7AeEb94ef92fe056C2f2A7', 'restricted': True, 'groupItemTitle': '↑ 100,000', 'groupItemThreshold': '13', 
'questionID': '0x3c9be67d4b90291760ac3bffc1f9470a1966e5c1f3e99131333170e3469bd023', 'enableOrderBook': True, 'orderPriceMinTickSize': 0.01, 'orderMinSize': 5, 'volumeNum': 2370982.5583320004, 'liquidityNum': 94083.1351, 'endDateIso': '2027-01-01', 
'startDateIso': '2025-11-24', 'hasReviewedDates': True, 'volume24hr': 1365.610655, 'volume1wk': 66640.791618, 'volume1mo': 228733.97684700004, 'volume1yr': 2370982.5583320004, 
'clobTokenIds': '["56078938060096976448086754249497300447360333783952000147427828224794011030104", "11291662904897713174667903388388696640643610556195928998276904135282270136756"]', 
'comboStatus': 'disabled', 'umaBond': '500', 'umaReward': '5', 'volume24hrClob': 1365.610655, 'volume1wkClob': 66640.791618, 'volume1moClob': 228733.97684700004, 'volume1yrClob': 2370982.5583320004, 'volumeClob': 2370982.5583320004, 
'liquidityClob': 94083.1351, 'makerBaseFee': 1000, 'takerBaseFee': 1000, 'customLiveness': 0, 'acceptingOrders': True, 'negRisk': False, 'negRiskRequestID': '', 
'events': [{'id': '89502', 'ticker': 'what-price-will-bitcoin-hit-before-2027', 'slug': 'what-price-will-bitcoin-hit-before-2027', 
            'title': 'What price will Bitcoin hit in 2026?', 'description': 'What price will Bitcoin hit before 2027?  ', 
            'resolutionSource': '', 'startDate': '2025-11-24T19:07:12.848Z', 'creationDate': '2025-11-24T19:13:13.705687Z', 'endDate': '2027-01-01T05:00:00Z', 
            'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
            'active': True, 'closed': False, 'archived': False, 'new': False, 'featured': False, 'restricted': True, 'liquidity': 2695251.24076, 'volume': 50935482.262915, 
            'openInterest': 9503925.568983998, 'createdAt': '2025-11-24T18:55:05.597959Z', 'updatedAt': '2026-08-02T10:55:09.654318Z', 'competitive': 0.9999750006249843, 
            'volume24hr': 130499.32620200001, 'volume1wk': 2062901.9714630004, 'volume1mo': 7099616.726362999, 'volume1yr': 49102712.92022599, 'enableOrderBook': True, 
            'liquidityClob': 2695251.24076, 'negRisk': False, 'commentCount': 0, 'series': [{'id': '10016', 'ticker': 'bitcoin-hit-price-monthly', 'slug': 'bitcoin-hit-price-monthly', 
                                                                                        'title': 'Bitcoin Hit Price Monthly', 'seriesType': 'single', 'recurrence': 'monthly', 
                                                                                        'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'active': True, 'closed': False, 'archived': False, 'featured': False, 'restricted': True, 
                                                                                        'createdAt': '2025-01-31T22:03:50.00441Z', 'updatedAt': '2026-08-02T10:55:28.185077Z', 
                                                                                        'volume24hr': 601000.637578, 'volume': 51492794.133888, 'liquidity': 3438609.2653, 'commentCount': 6318, 
                                                                                        'requiresTranslation': False}], 
            'cyom': False, 'showAllOutcomes': True, 'showMarketImages': False, 'enableNegRisk': False, 'automaticallyActive': True, 'seriesSlug': 'bitcoin-hit-price-monthly', 
            'gmpChartMode': 'default', 'negRiskAugmented': False, 'estimateValue': True, 'cantEstimate': True, 'cumulativeMarkets': False, 'pendingDeployment': False, 'deploying': False, 
            'requiresTranslation': False, 'eventMetadata': {'context_requires_regen': True}, 'version': 'v1'}], 

'ready': False, 'funded': False, 'acceptingOrdersTimestamp': '2025-11-24T19:06:55Z', 
'cyom': False, 'competitive': 0.8590880780051975, 'pagerDutyNotificationEnabled': False, 'approved': True, 'clobRewards': [{'id': '418394', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 
                                                                                                                        'assetAddress': '0xc011a7e12a19f7b1f670d46f03b03f3342e82dfb', 'rewardsAmount': 0, 'rewardsDailyRate': 0.001, 
                                                                                                                        'startDate': '2026-06-03', 'endDate': '2500-12-31'}], 
'rewardsMinSize': 0, 'rewardsMaxSpread': 0, 'spread': 0.01, 'oneMonthPriceChange': -0.01, 'lastTradePrice': 0.09, 'bestBid': 0.09, 'bestAsk': 0.1, 'automaticallyActive': True, 
'clearBookOnStart': True, 'seriesColor': '', 'showGmpSeries': False, 'showGmpOutcome': False, 'manualActivation': False, 'negRiskOther': False, 'umaResolutionStatuses': '[]', 
'pendingDeployment': False, 'deploying': False, 'deployingTimestamp': '2025-11-24T19:06:23.727362Z', 'rfqEnabled': False, 'holdingRewardsEnabled': True, 'feesEnabled': True, 
'requiresTranslation': False, 'feeType': 'crypto_fees_v2', 'feeSchedule': {'exponent': 1, 'rate': 0.07, 'takerOnly': True, 'rebateRate': 0.2}, 'version': 'v1'}
["0.095", "0.905"]